# Train a Tiny GPT from Scratch on MetaX C550 (64 GiB)

This notebook is an educational, single-GPU GPT-style pretraining run. It trains a decoder-only Transformer from random initialization on public WikiText text. It is **not** fine-tuning a pretrained model.

## Hardware and compatibility notes

- Target: MetaX C550, 64 GiB, using the site's installed **MACA-enabled PyTorch environment**.
- Do not replace the vendor-provided PyTorch/MACA packages with a generic PyTorch wheel. Confirm the environment with your platform's MACA installation guide.
- MACA compatibility can differ by release. This notebook starts with ordinary PyTorch operations and a conservative FP32 path; it probes mixed precision before enabling it. It does not require FlashAttention, `torch.compile`, or CUDA-specific fused optimizers.
- The device is expected to expose the usual `torch.cuda`-compatible interface in a MACA-enabled build. If `torch.cuda.is_available()` is false, stop and fix the environment before training.
- Default configuration is intentionally modest (~30M parameters with the default 8,000-token vocabulary): 8 layers, width 512, 8 heads, 512-token context. It should fit comfortably in 64 GiB, but actual speed depends heavily on MACA/PyTorch versions and kernels.
- For a short run (3,000 steps took about ten minutes on one C550), the goal is to learn the training mechanics and observe loss decrease—not to produce a useful general-purpose assistant. WikiText-103 is small by modern pretraining standards, and a short run will be undertrained.

Suggested environment packages, installed in the notebook's kernel:
`datasets`, `tokenizers`, `tqdm`, `matplotlib`, `safetensors`.

## Prerequisites in a CubeStack DevEnvironment

DevEnvironment egress is default-deny, and no field in the DevEnvironment spec
widens it (cubestack#312). Two sections below reach the network, so both have to
be reachable before you start:

- **§1** runs `%pip install`. `%pip` reports its own progress, not its failure: a
  blocked install surfaces much later, as a `ModuleNotFoundError` from whichever
  cell first imports what it installed.
- **§4** downloads `Salesforce/wikitext` from huggingface.co. Only the first run
  needs this — the download cache in `~/.cache/huggingface` is shared, so a re-run
  reads it from disk and never notices whether egress works.

The remaining sections are compute-only.


## 1. Install Python-level dependencies

Run this only if these packages are missing. Do not install or upgrade `torch` here; use the MACA-compatible build supplied for your server.

In [ ]:
# Versions are pinned to ones this image supports: the image ships torch 2.4
# (vendor build), and newer releases break against it. datasets 5.x pulls
# httpx2, which calls Brotli's process(output_buffer_limit=...) -- an argument
# the image's Brotli 1.1.0 rejects -- and huggingface-hub 1.x conflicts with the
# image's transformers 4.57.6 (which requires <1.0). Do not drop the pins or
# add -U.
%pip install datasets==4.3.0 tokenizers==0.22.2 huggingface-hub==0.36.2 tqdm==4.67.1 matplotlib==3.10.9 safetensors==0.8.0

print("✓ Cell 1 complete — dependencies installed.")

## 2. Verify the GPU and PyTorch/MACA environment

If this cell fails, resolve the MACA/PyTorch installation first. Keep the printed versions with your experiment notes.

In [ ]:
import os, sys, platform, time, math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("torch.cuda available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "No accelerator exposed through torch.cuda. This notebook expects the "
        "MACA-enabled PyTorch environment for MetaX C550. Check the vendor "
        "runtime, environment variables, and installation instructions."
    )

print("Device name:", torch.cuda.get_device_name(0))
print("Device properties:", torch.cuda.get_device_properties(0))
try:
    print("BF16 reported supported:", torch.cuda.is_bf16_supported())
except Exception as e:
    print("BF16 capability probe unavailable:", repr(e))

device = torch.device("cuda:0")

print(f"✓ Cell 2 complete — MACA runtime verified on {torch.cuda.get_device_name(0)}.")

## 3. Configure a short educational run

Start with these settings. If performance is good and memory use is low, increase `batch_size` or `max_steps` gradually. If an out-of-memory error occurs, reduce `batch_size` first, then `block_size`.

We use a BPE tokenizer with a 8,000-token vocabulary, an 8-layer decoder, and a 512-token context. Smaller vocabulary makes the first experiment faster while still demonstrating subword tokenization.

In [ ]:
import json

SEED = 1337
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Model
VOCAB_SIZE = 8000
BLOCK_SIZE = 512
N_LAYER = 8
N_HEAD = 8
N_EMBD = 512
DROPOUT = 0.1

# Training
BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 4
MAX_STEPS = 3000
EVAL_INTERVAL = 200
EVAL_BATCHES = 20
LEARNING_RATE = 3e-4
MIN_LR = 3e-5
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.1
GRAD_CLIP = 1.0

DATASET_CONFIG = "wikitext-103-raw-v1"
# Limit the training text to keep preprocessing and the experiment manageable.
# Set to None to tokenize the entire split (more RAM and preprocessing time).
MAX_TRAIN_CHARS = 8_000_000
MAX_VALID_CHARS = 500_000

# Each execution gets its own directory under ~/runs, so a run never overwrites
# an earlier one. Everything this notebook writes -- the tokenized data, the
# tokenizer, the checkpoints and the exported model -- is derived from RUN_DIR.
# The Hugging Face *download* cache stays shared (~/.cache/huggingface) so a
# re-run does not download WikiText-103 again.
#
# To continue an earlier run, set RESUME_RUN_ID to its directory name (for
# example "20261016-050130") and re-run the notebook from here; every path below
# then resolves inside that run instead of a new one. Section 16 walks through
# the full procedure. Leave it None for a fresh run.
RESUME_RUN_ID = None
RUN_ID = RESUME_RUN_ID or time.strftime("%Y%m%d-%H%M%S")
RUN_DIR = os.path.join(os.path.expanduser("~"), "runs", RUN_ID)
DATA_DIR = os.path.join(RUN_DIR, "data")
TOKENIZER_DIR = os.path.join(RUN_DIR, "tokenizer")
CHECKPOINT_DIR = os.path.join(RUN_DIR, "checkpoints")
SERVING_DIR = os.path.join(RUN_DIR, "serving")
for _d in (DATA_DIR, TOKENIZER_DIR, CHECKPOINT_DIR, SERVING_DIR):
    os.makedirs(_d, exist_ok=True)

run_record = {
    "run_id": RUN_ID,
    "run_dir": RUN_DIR,
    "seed": SEED,
    "model": {
        "vocab_size": VOCAB_SIZE,
        "block_size": BLOCK_SIZE,
        "n_layer": N_LAYER,
        "n_head": N_HEAD,
        "n_embd": N_EMBD,
        "dropout": DROPOUT,
    },
    "training": {
        "batch_size": BATCH_SIZE,
        "grad_accum_steps": GRAD_ACCUM_STEPS,
        "max_steps": MAX_STEPS,
        "eval_interval": EVAL_INTERVAL,
        "learning_rate": LEARNING_RATE,
        "min_lr": MIN_LR,
        "warmup_steps": WARMUP_STEPS,
        "weight_decay": WEIGHT_DECAY,
        "grad_clip": GRAD_CLIP,
    },
    "data": {
        "dataset": "Salesforce/wikitext:" + DATASET_CONFIG,
        "max_train_chars": MAX_TRAIN_CHARS,
        "max_valid_chars": MAX_VALID_CHARS,
    },
    "environment": {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "device": torch.cuda.get_device_name(0),
    },
}
with open(os.path.join(RUN_DIR, "run.json"), "w", encoding="utf-8") as f:
    json.dump(run_record, f, indent=2)

print("Run directory:", RUN_DIR)
print("Config ready.")

print(f"✓ Cell 3 complete — configuration set for run {RUN_ID} ({MAX_STEPS} steps).")

## 4. Download public text data

We use the raw WikiText-103 dataset through Hugging Face Datasets. The train split is used to train the tokenizer and model; the validation split is kept separate for evaluation. If downloading WikiText-103 is inconvenient, change `DATASET_CONFIG` to `"wikitext-2-raw-v1"` in the configuration cell above.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("Salesforce/wikitext", DATASET_CONFIG)

def join_wikitext_split(split, max_chars=None):
    # Keep line boundaries, skip empty lines. Raw WikiText includes article
    # markers and punctuation, which are useful for a language-modeling demo.
    parts = [line for line in split["text"] if line and line.strip()]
    text = "\n\n".join(parts)
    if max_chars is not None:
        text = text[:max_chars]
    return text

train_text = join_wikitext_split(dataset["train"], MAX_TRAIN_CHARS)
valid_text = join_wikitext_split(dataset["validation"], MAX_VALID_CHARS)

# The text this run was built from is part of the run directory, so the working
# directory stays self-contained even if the HF cache is later evicted.
for _name, _text in (("train.txt", train_text), ("valid.txt", valid_text)):
    with open(os.path.join(DATA_DIR, _name), "w", encoding="utf-8") as f:
        f.write(_text)
print("Wrote train.txt / valid.txt to", DATA_DIR)

print("Train characters:", f"{len(train_text):,}")
print("Validation characters:", f"{len(valid_text):,}")
print("\nSample:\n", train_text[:700])

print(f"✓ Cell 4 complete — WikiText written ({len(train_text):,} train / {len(valid_text):,} validation characters).")

## 5. Train a BPE tokenizer from scratch

This learns the tokenizer from the training text only, avoiding validation leakage. A byte-level BPE tokenizer is a common practical choice. We add an EOS token to mark document boundaries.

In [ ]:
from tokenizers import ByteLevelBPETokenizer

TRAIN_TEXT_PATH = os.path.join(DATA_DIR, "train.txt")
VOCAB_PATH = os.path.join(TOKENIZER_DIR, "vocab.json")
MERGES_PATH = os.path.join(TOKENIZER_DIR, "merges.txt")

# When resuming, RUN_DIR already holds the tokenizer this run was built with.
# Retraining it would be wasted work and could produce a different vocabulary,
# which would invalidate every checkpoint in that directory.
if os.path.exists(VOCAB_PATH) and os.path.exists(MERGES_PATH):
    print("Reusing tokenizer in", TOKENIZER_DIR)
else:
    tokenizer = ByteLevelBPETokenizer()
    tokenizer.train(
        files=[TRAIN_TEXT_PATH],
        vocab_size=VOCAB_SIZE,
        min_frequency=2,
        special_tokens=["<|endoftext|>"],
    )
    tokenizer.save_model(TOKENIZER_DIR)

# Load from the saved files so those files, not the in-memory object, are the
# source of truth.
tokenizer = ByteLevelBPETokenizer(VOCAB_PATH, MERGES_PATH)
tokenizer.add_special_tokens(["<|endoftext|>"])
EOS_ID = tokenizer.token_to_id("<|endoftext|>")
print("Tokenizer vocabulary:", tokenizer.get_vocab_size())
print("EOS token ID:", EOS_ID)
print("Round-trip:", tokenizer.decode(tokenizer.encode("A tiny GPT learns.").ids))

print(f"✓ Cell 5 complete — BPE tokenizer trained ({tokenizer.get_vocab_size():,} tokens).")

## 6. Encode text into token IDs

Each non-empty WikiText line is treated as a unit, and the code appends one EOS marker after it, so a document boundary is a real boundary rather than a fixed offset in the text. The lines are encoded in batches to avoid one very large tokenizer call. The token arrays live in host RAM; batches are copied to the GPU.

In [ ]:
def encode_text(text, batch_size=1024):
    # Section 4 joined the non-empty WikiText lines with a blank line, so
    # splitting on "\n\n" recovers exactly those units. EOS then marks a real
    # line/document boundary instead of an arbitrary offset in the character
    # stream. Encoding in batches keeps any single tokenizer call small.
    parts = text.split("\n\n")
    ids = []
    for start in range(0, len(parts), batch_size):
        for encoding in tokenizer.encode_batch(parts[start:start + batch_size]):
            ids.extend(encoding.ids)
            ids.append(EOS_ID)
    return np.asarray(ids, dtype=np.uint32)

train_ids = encode_text(train_text)
valid_ids = encode_text(valid_text)

np.save(os.path.join(DATA_DIR, "train_ids.npy"), train_ids)
np.save(os.path.join(DATA_DIR, "valid_ids.npy"), valid_ids)

print("Train tokens:", f"{len(train_ids):,}")
print("Validation tokens:", f"{len(valid_ids):,}")
print("Approximate training token batches/step:", BATCH_SIZE * GRAD_ACCUM_STEPS * BLOCK_SIZE)
if len(train_ids) <= BLOCK_SIZE + 1 or len(valid_ids) <= BLOCK_SIZE + 1:
    raise ValueError("Dataset too small for BLOCK_SIZE; reduce BLOCK_SIZE or load more text.")

print(f"✓ Cell 6 complete — text encoded ({len(train_ids):,} train / {len(valid_ids):,} validation tokens).")

## 7. Efficient random sequence batches

Each training example contains a context of `BLOCK_SIZE` tokens. The target is shifted by one token, so the model learns next-token prediction. Random windows make it easy to iterate over a text stream without materializing every overlapping sequence.

In [ ]:
def get_batch(split, batch_size=BATCH_SIZE):
    source = train_ids if split == "train" else valid_ids
    # int64 is required by embedding / cross_entropy indexing.
    ix = np.random.randint(0, len(source) - BLOCK_SIZE - 1, size=(batch_size,))
    x = np.stack([source[i:i + BLOCK_SIZE] for i in ix]).astype(np.int64)
    y = np.stack([source[i + 1:i + BLOCK_SIZE + 1] for i in ix]).astype(np.int64)
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

xb, yb = get_batch("train", batch_size=2)
print("x:", tuple(xb.shape), "y:", tuple(yb.shape))
print("Input tokens:", tokenizer.decode(xb[0, :30].tolist()))
print("Target tokens:", tokenizer.decode(yb[0, :30].tolist()))

print(f"✓ Cell 7 complete — random batches ready (x {tuple(xb.shape)}, y {tuple(yb.shape)}).")

## 8. Implement the decoder-only Transformer

For attention we call `torch.nn.functional.scaled_dot_product_attention`. This is standard PyTorch API, but the actual optimized kernel available depends on the installed backend. On a MACA build it may use a vendor implementation or a generic fallback; this notebook does not assume NVIDIA FlashAttention is available.

Architecture:
- learned token and positional embeddings
- pre-LayerNorm Transformer blocks
- causal multi-head self-attention
- GELU feed-forward network with 4× expansion
- residual connections
- output projection to vocabulary logits

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=True)
        self.proj = nn.Linear(n_embd, n_embd, bias=True)
        self.attn_dropout = dropout
        self.resid_dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=-1)

        def reshape_heads(t):
            return t.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        q, k, v = map(reshape_heads, (q, k, v))
        # Dropout is only active during training.
        out = F.scaled_dot_product_attention(
            q, k, v,
            attn_mask=None,
            dropout_p=self.attn_dropout if self.training else 0.0,
            is_causal=True,
        )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.proj(out))


class MLP(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class TinyGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer, n_head, n_embd, dropout):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[
            Block(n_embd, n_head, dropout) for _ in range(n_layer)
        ])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        # Weight tying is common in small GPT language models.
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} exceeds block_size {self.block_size}")
        pos = torch.arange(T, device=idx.device)
        x = self.drop(self.token_emb(idx) + self.pos_emb(pos)[None, :, :])
        x = self.blocks(x)
        logits = self.lm_head(self.ln_f(x))

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                targets.reshape(-1),
            )
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=100, temperature=0.8, top_k=50):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            if top_k is not None:
                k = min(top_k, logits.size(-1))
                values, _ = torch.topk(logits, k)
                logits[logits < values[:, [-1]]] = -float("inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
        return idx

print("✓ Cell 8 complete — Transformer blocks defined.")

## 9. Instantiate the model and smoke-test a forward/backward pass

This verifies basic operator support before spending time training. If this fails inside SDPA or another operation, keep the traceback and check the exact MACA PyTorch release's operator support. As a fallback, the attention implementation can be replaced with explicit QKᵀ + causal mask + softmax, but that may use more memory and run slower.

In [ ]:
model = TinyGPT(
    vocab_size=tokenizer.get_vocab_size(),
    block_size=BLOCK_SIZE,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_embd=N_EMBD,
    dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,} ({n_params / 1e6:.1f}M)")

model.train()
test_logits, test_loss = model(xb, yb)
test_loss.backward()
model.zero_grad(set_to_none=True)
print("Forward/backward OK; loss:", float(test_loss.detach().cpu()))
print("Allocated memory (GiB):", round(torch.cuda.memory_allocated() / 1024**3, 3))

print(f"✓ Cell 9 complete — model built ({n_params:,} parameters).")

## 10. Probe mixed precision

MACA releases may differ in BF16/FP16 support and stability. The probe below tests a small autocast forward/backward pass. It prefers BF16 only when the runtime reports support and the actual probe succeeds; otherwise it tries FP16 with GradScaler. If both fail, it falls back to FP32.

Do not assume a precision is supported based only on the GPU model. The test uses the installed runtime and operators.

In [ ]:
def try_amp(dtype):
    try:
        m = TinyGPT(
            vocab_size=tokenizer.get_vocab_size(),
            block_size=BLOCK_SIZE,
            n_layer=2, n_head=N_HEAD, n_embd=N_EMBD, dropout=0.0,
        ).to(device)
        opt = torch.optim.AdamW(m.parameters(), lr=1e-4)
        with torch.autocast(device_type="cuda", dtype=dtype, enabled=True):
            _, loss = m(xb, yb)
        loss.backward()
        opt.zero_grad(set_to_none=True)
        del m, opt, loss
        torch.cuda.empty_cache()
        return True
    except Exception as e:
        print(f"{dtype} probe failed:", repr(e)[:500])
        torch.cuda.empty_cache()
        return False

AMP_DTYPE = None
if getattr(torch.cuda, "is_bf16_supported", lambda: False)():
    if try_amp(torch.bfloat16):
        AMP_DTYPE = torch.bfloat16

if AMP_DTYPE is None and try_amp(torch.float16):
    AMP_DTYPE = torch.float16

USE_AMP = AMP_DTYPE is not None
USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16

print("Mixed precision enabled:", USE_AMP)
print("Autocast dtype:", AMP_DTYPE)
print("GradScaler enabled:", USE_SCALER)

print(f"✓ Cell 10 complete — precision probe finished (mixed precision: {USE_AMP}).")

## 11. Training utilities

The evaluation function reports mean cross-entropy on separate validation text. Perplexity is `exp(loss)` and can overflow for very high losses, so we guard it. The learning rate uses linear warmup followed by cosine decay.

In [ ]:
@torch.no_grad()
def estimate_loss():
    model.eval()
    results = {}
    for split in ("train", "val"):
        losses = []
        for _ in range(EVAL_BATCHES):
            xb, yb = get_batch("train" if split == "train" else "val")
            with torch.autocast(
                device_type="cuda",
                dtype=AMP_DTYPE if USE_AMP else torch.float32,
                enabled=USE_AMP,
            ):
                _, loss = model(xb, yb)
            losses.append(float(loss.detach().float().cpu()))
        results[split] = float(np.mean(losses))
    model.train()
    return results

def get_lr(step):
    if step < WARMUP_STEPS:
        return LEARNING_RATE * (step + 1) / max(1, WARMUP_STEPS)
    progress = (step - WARMUP_STEPS) / max(1, MAX_STEPS - WARMUP_STEPS)
    progress = min(max(progress, 0.0), 1.0)
    return MIN_LR + 0.5 * (LEARNING_RATE - MIN_LR) * (1 + math.cos(math.pi * progress))

scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    betas=(0.9, 0.95),
    weight_decay=WEIGHT_DECAY,
)
print("Training utilities ready.")

print("✓ Cell 11 complete — training utilities ready.")

## 12. Train, validate, and save checkpoints

A checkpoint includes model weights, optimizer state, step, configuration, and tokenizer files are saved separately in the run's `tokenizer/` directory. Save checkpoints periodically so an interrupted notebook can be resumed.

If MACA's GradScaler implementation is unavailable, set `USE_SCALER = False`; FP16 may then be less numerically robust. If mixed precision is unstable, set `USE_AMP = False` to use FP32.

In [ ]:
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

history = {"step": [], "train": [], "val": []}
checkpoint_path = os.path.join(CHECKPOINT_DIR, "latest.pt")
best_path = os.path.join(CHECKPOINT_DIR, "best.pt")
best_val = float("inf")

def save_checkpoint(path, step, val_loss):
    payload = {
        "step": step,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "val_loss": val_loss,
        "config": {
            "vocab_size": tokenizer.get_vocab_size(),
            "block_size": BLOCK_SIZE,
            "n_layer": N_LAYER,
            "n_head": N_HEAD,
            "n_embd": N_EMBD,
            "dropout": DROPOUT,
            "amp_dtype": str(AMP_DTYPE),
            "use_amp": USE_AMP,
        },
    }
    torch.save(payload, path)

start_step = 0
if RESUME_RUN_ID is not None:
    # Section 3 pointed RUN_DIR at an existing run, so continue that run rather
    # than starting over and overwriting its checkpoints. This has to happen
    # here, before the loop: step 0 evaluates, saves and -- since best_val
    # starts at infinity -- would replace best.pt with a fresh model.
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"No checkpoint to resume from: {checkpoint_path}")
    ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    try:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    except Exception as e:
        print("Scaler state not restored:", e)
    start_step = ckpt["step"] + 1
    if os.path.exists(best_path):
        best_val = torch.load(best_path, map_location="cpu", weights_only=False)["val_loss"]
    print("Resuming from step:", start_step, "best validation loss:", round(best_val, 4))

model.train()
optimizer.zero_grad(set_to_none=True)
start_time = time.time()

pbar = tqdm(range(start_step, MAX_STEPS), desc="training")
for step in pbar:
    lr = get_lr(step)
    for pg in optimizer.param_groups:
        pg["lr"] = lr

    accum_loss = 0.0
    for micro_step in range(GRAD_ACCUM_STEPS):
        xb, yb = get_batch("train")
        with torch.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE if USE_AMP else torch.float32,
            enabled=USE_AMP,
        ):
            _, loss = model(xb, yb)
            loss = loss / GRAD_ACCUM_STEPS

        if USE_SCALER:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        accum_loss += float(loss.detach().float().cpu())

    if USE_SCALER:
        scaler.unscale_(optimizer)

    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)

    if USE_SCALER:
        scaler.step(optimizer)
        scaler.update()
    else:
        optimizer.step()
    optimizer.zero_grad(set_to_none=True)

    if step % EVAL_INTERVAL == 0 or step == MAX_STEPS - 1:
        metrics = estimate_loss()
        history["step"].append(step)
        history["train"].append(metrics["train"])
        history["val"].append(metrics["val"])
        elapsed = time.time() - start_time
        pbar.set_postfix(
            loss=f"{accum_loss:.3f}",
            val=f"{metrics['val']:.3f}",
            lr=f"{lr:.2e}",
            minutes=f"{elapsed/60:.1f}",
        )
        save_checkpoint(checkpoint_path, step, metrics["val"])
        if metrics["val"] < best_val:
            best_val = metrics["val"]
            save_checkpoint(best_path, step, best_val)

print(f"Finished in {(time.time() - start_time)/60:.1f} minutes")

print(f"✓ Cell 12 complete — trained for {MAX_STEPS} steps.")

## 13. Plot training and validation loss

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(history["step"], history["train"], marker="o", label="Train loss")
plt.plot(history["step"], history["val"], marker="o", label="Validation loss")
plt.xlabel("Training step")
plt.ylabel("Cross-entropy loss (nats/token)")
plt.title("Tiny GPT pretraining on WikiText")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

if history["val"]:
    final_val = history["val"][-1]
    print("Final validation loss:", round(final_val, 4))
    print("Final validation perplexity:", round(math.exp(min(final_val, 20)), 2))

print(f"✓ Cell 13 complete — loss plot rendered ({len(history['step'])} evaluations).")

## 14. Generate text

The model is trained from random initialization, so early outputs may look noisy or repetitive. Generation quality depends on how many useful tokens it sees, not just the parameter count.

In [ ]:
model.eval()
prompt = "The history of"
prompt_ids = tokenizer.encode(prompt).ids
if not prompt_ids:
    prompt_ids = [EOS_ID]

context = torch.tensor([prompt_ids], dtype=torch.long, device=device)
with torch.no_grad():
    generated = model.generate(
        context,
        max_new_tokens=200,
        temperature=0.8,
        top_k=50,
    )

print(tokenizer.decode(generated[0].tolist()))

print(f"✓ Cell 14 complete — generated {generated.shape[1] - len(prompt_ids)} tokens from the prompt.")

## 15. Export a servable Hugging Face model

The checkpoint above is a `state_dict` for the `TinyGPT` class defined in this notebook, so nothing
outside this notebook can load the result. This cell rewrites it as a standard Hugging Face **GPT-2**
directory, which `transformers` and vLLM both understand.

That works because `TinyGPT` is a GPT-2 layer for layer: pre-norm blocks, a fused q|k|v projection, a
4x GELU MLP, learned absolute positions, and a language-model head tied to the token embedding. Only
the names differ, along with the layout of the projection weights — GPT-2 keeps them in `Conv1D`,
which is input-major, where `nn.Linear` is output-major.

Two things worth knowing about the result:

- `eos_token_id` is the `<|endoftext|>` id, and the tokenizer carries no post-processor — matching how
  the training text was encoded, so a prompt tokenizes here exactly as it did then.
- The model is servable, not capable. It has seen a few thousand steps of WikiText and nothing else.


In [ ]:
import json, shutil
from safetensors.torch import save_file

checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
sd = checkpoint["model_state_dict"]
model_vocab_size = tokenizer.get_vocab_size()

def as_conv1d(weight):
    # nn.Linear stores weight as (out, in); transformers' Conv1D as (in, out).
    return weight.t().contiguous()

# The fused q|k|v projection needs no splitting: TinyGPT builds it as one
# Linear over q, k, v in that order, which is the order GPT-2's c_attn reads
# as well, so the transpose alone places all three.
export = {
    "transformer.wte.weight": sd["token_emb.weight"].contiguous(),
    "transformer.wpe.weight": sd["pos_emb.weight"].contiguous(),
    "transformer.ln_f.weight": sd["ln_f.weight"].contiguous(),
    "transformer.ln_f.bias": sd["ln_f.bias"].contiguous(),
}
for i in range(N_LAYER):
    src, dst = f"blocks.{i}.", f"transformer.h.{i}."
    export[dst + "ln_1.weight"] = sd[src + "ln1.weight"].contiguous()
    export[dst + "ln_1.bias"] = sd[src + "ln1.bias"].contiguous()
    export[dst + "ln_2.weight"] = sd[src + "ln2.weight"].contiguous()
    export[dst + "ln_2.bias"] = sd[src + "ln2.bias"].contiguous()
    export[dst + "attn.c_attn.weight"] = as_conv1d(sd[src + "attn.qkv.weight"])
    export[dst + "attn.c_attn.bias"] = sd[src + "attn.qkv.bias"].contiguous()
    export[dst + "attn.c_proj.weight"] = as_conv1d(sd[src + "attn.proj.weight"])
    export[dst + "attn.c_proj.bias"] = sd[src + "attn.proj.bias"].contiguous()
    export[dst + "mlp.c_fc.weight"] = as_conv1d(sd[src + "mlp.net.0.weight"])
    export[dst + "mlp.c_fc.bias"] = sd[src + "mlp.net.0.bias"].contiguous()
    export[dst + "mlp.c_proj.weight"] = as_conv1d(sd[src + "mlp.net.2.weight"])
    export[dst + "mlp.c_proj.bias"] = sd[src + "mlp.net.2.bias"].contiguous()
# lm_head is tied to the token embedding, so it is named in config.json rather
# than stored a second time -- as it is in a released GPT-2 checkpoint.

config = {
    "architectures": ["GPT2LMHeadModel"],
    "model_type": "gpt2",
    "vocab_size": model_vocab_size,
    "n_positions": BLOCK_SIZE,
    "n_ctx": BLOCK_SIZE,
    "n_embd": N_EMBD,
    "n_layer": N_LAYER,
    "n_head": N_HEAD,
    "n_inner": 4 * N_EMBD,
    "activation_function": "gelu",
    "resid_pdrop": DROPOUT,
    "embd_pdrop": DROPOUT,
    "attn_pdrop": DROPOUT,
    "layer_norm_epsilon": 1e-5,
    "initializer_range": 0.02,
    "tie_word_embeddings": True,
    "bos_token_id": EOS_ID,
    "eos_token_id": EOS_ID,
}
with open(os.path.join(SERVING_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

save_file(export, os.path.join(SERVING_DIR, "model.safetensors"), metadata={"format": "pt"})

# This tokenizer has no post-processor, which matches how the training text was
# encoded: a prompt is tokenized here exactly as it was then.
tokenizer.save(os.path.join(SERVING_DIR, "tokenizer.json"))
for name in ("vocab.json", "merges.txt"):
    shutil.copy(os.path.join(TOKENIZER_DIR, name), SERVING_DIR)

tokenizer_config = {
    "tokenizer_class": "PreTrainedTokenizerFast",
    "model_max_length": BLOCK_SIZE,
    "bos_token": "<|endoftext|>",
    "eos_token": "<|endoftext|>",
    "unk_token": "<|endoftext|>",
    "add_bos_token": False,
    "add_eos_token": False,
    "clean_up_tokenization_spaces": False,
}
with open(os.path.join(SERVING_DIR, "tokenizer_config.json"), "w", encoding="utf-8") as f:
    json.dump(tokenizer_config, f, indent=2)

special_tokens_map = {
    "bos_token": "<|endoftext|>",
    "eos_token": "<|endoftext|>",
    "unk_token": "<|endoftext|>",
}
with open(os.path.join(SERVING_DIR, "special_tokens_map.json"), "w", encoding="utf-8") as f:
    json.dump(special_tokens_map, f, indent=2)

print("Exported to", SERVING_DIR)
for name in sorted(os.listdir(SERVING_DIR)):
    print("  %-26s %9.1f KiB" % (name, os.path.getsize(os.path.join(SERVING_DIR, name)) / 1024))

print(f"✓ Cell 15a complete — model exported to {SERVING_DIR}.")

In [ ]:
# A transposed weight is easy to write and hard to notice, so undo the rename
# and check the model still produces the same logits. Going back through
# TinyGPT rather than a second GPT-2 implementation means this compares the
# export against the model that was actually trained.
#
# The tensors are read back from model.safetensors rather than reused from the
# in-memory `export` dict: a check that never reads the file cannot notice a
# bad write or a missing entry.
from safetensors.torch import load_file

written = load_file(os.path.join(SERVING_DIR, "model.safetensors"))
reloaded = {
    "token_emb.weight": written["transformer.wte.weight"],
    "pos_emb.weight": written["transformer.wpe.weight"],
    "ln_f.weight": written["transformer.ln_f.weight"],
    "ln_f.bias": written["transformer.ln_f.bias"],
    "lm_head.weight": written["transformer.wte.weight"],
}
for i in range(N_LAYER):
    dst, src = f"blocks.{i}.", f"transformer.h.{i}."
    reloaded[dst + "ln1.weight"] = written[src + "ln_1.weight"]
    reloaded[dst + "ln1.bias"] = written[src + "ln_1.bias"]
    reloaded[dst + "ln2.weight"] = written[src + "ln_2.weight"]
    reloaded[dst + "ln2.bias"] = written[src + "ln_2.bias"]
    reloaded[dst + "attn.qkv.weight"] = written[src + "attn.c_attn.weight"].t().contiguous()
    reloaded[dst + "attn.qkv.bias"] = written[src + "attn.c_attn.bias"]
    reloaded[dst + "attn.proj.weight"] = written[src + "attn.c_proj.weight"].t().contiguous()
    reloaded[dst + "attn.proj.bias"] = written[src + "attn.c_proj.bias"]
    reloaded[dst + "mlp.net.0.weight"] = written[src + "mlp.c_fc.weight"].t().contiguous()
    reloaded[dst + "mlp.net.0.bias"] = written[src + "mlp.c_fc.bias"]
    reloaded[dst + "mlp.net.2.weight"] = written[src + "mlp.c_proj.weight"].t().contiguous()
    reloaded[dst + "mlp.net.2.bias"] = written[src + "mlp.c_proj.bias"]

roundtrip = TinyGPT(
    vocab_size=model_vocab_size,
    block_size=BLOCK_SIZE,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    n_embd=N_EMBD,
    dropout=DROPOUT,
).to(device)
roundtrip.load_state_dict(reloaded)
roundtrip.eval()

model.eval()
xb_check, _ = get_batch("train", batch_size=4)
with torch.no_grad():
    trained_logits, _ = model(xb_check)
    exported_logits, _ = roundtrip(xb_check)

delta = float((trained_logits - exported_logits).abs().max().cpu())
print("Max logit difference after export and reload:", delta)
if delta >= 1e-4:
    raise RuntimeError("The exported weights do not reproduce the trained model.")
print("Export verified.")

# The directory claims to be a GPT-2 checkpoint. When `transformers` is
# importable, load it as one and compare against the same logits; that is the
# only check that exercises config.json and the Conv1D layout from the
# library's side.
#
# Only a failed import skips this check: a vendor PyTorch build can be older
# than transformers supports, and that failure can surface as something other
# than ImportError, so the import is caught broadly. Everything after it --
# loading the directory, reading the logits, comparing them -- is outside the
# try, because reporting a broken export as "skipped" is the one outcome this
# check exists to prevent.
try:
    from transformers import GPT2LMHeadModel
except Exception as e:
    print("transformers not importable; GPT-2 check skipped:", repr(e)[:300])
else:
    # This MACA build enables TF32 for matmul by default. With it on, the same
    # weights produce logits that differ from transformers' by ~5e-3 -- kernel
    # rounding, not an export error -- so turn it off and recompute both sides
    # for the comparison. 1e-4 then sits well above the observed float32 noise
    # and far below what a transposed or misnamed weight would produce.
    tf32_before = torch.backends.cuda.matmul.allow_tf32
    torch.backends.cuda.matmul.allow_tf32 = False
    try:
        hf = GPT2LMHeadModel.from_pretrained(SERVING_DIR).to(device).eval()
        with torch.no_grad():
            reference_logits, _ = model(xb_check)
            hf_logits = hf(xb_check).logits
        hf_delta = float((reference_logits - hf_logits).abs().max().cpu())
    finally:
        torch.backends.cuda.matmul.allow_tf32 = tf32_before
    print("Max logit difference vs transformers GPT2LMHeadModel:", hf_delta)
    if hf_delta >= 1e-4:
        raise RuntimeError("transformers does not reproduce the trained model.")

print(f"✓ Cell 15b complete — exported weights verified (max logit difference {delta:.2e}).")

## 16. Resume from the latest checkpoint

To continue an earlier run after restarting the kernel, set `RESUME_RUN_ID` in section 3 to that
run's directory name (the name under `~/runs/`) and re-run the notebook. Section 3 resolves every
path inside the existing run and section 5 reuses the tokenizer it already saved, so no cell needs
editing. Section 12 then loads `checkpoints/latest.pt` — model, optimizer and scaler — and continues
from the next step instead of starting over.

`best.pt` carries its own validation loss, so a resumed run only replaces it by beating that figure
rather than resetting the watermark. The loss history itself is not stored in a checkpoint, so
section 13 plots only the steps run after the resume.

## 17. Experiments to try next

1. **See attention internals:** capture the attention weights for one head. The current SDPA path does not return weights; temporarily replace it with explicit `q @ k.transpose(-2, -1)`, causal masking, softmax, and `weights @ v` for a short sequence.
2. **Change model size:** try 4 layers / width 256 for a fast baseline, then 8 layers / width 512. Keep the context length fixed while comparing.
3. **Change context length:** 256 vs 512. Longer context increases attention compute and memory.
4. **Ablation:** compare learned positional embeddings against sinusoidal embeddings, or remove dropout.
5. **Measure throughput:** report tokens/sec as `BATCH_SIZE * GRAD_ACCUM_STEPS * BLOCK_SIZE / step_seconds`, and compare settings only after warm-up.
6. **Keep a real validation split:** don't train the tokenizer or model on validation text.

### Limitations

- WikiText is a relatively small, Wikipedia-style corpus. A few hours of training will teach the model local language patterns but not broad world knowledge or instruction following.
- The BPE tokenizer is trained on a capped subset of training text. This is sufficient for the experiment, not an optimal production tokenizer.
- Kernel availability and performance on MetaX C550 depend on the installed MACA/PyTorch release. If SDPA is unsupported, use a manual attention implementation for learning, or a vendor-supported PyTorch build.
- `torch.cuda` API compatibility is assumed by this notebook. Do not interpret a successful import as proof that every CUDA-specific kernel is available on MACA.
- The notebook prioritizes readability over maximum throughput. It does not use distributed training, FlashAttention-specific packages, `torch.compile`, or vendor-specific fused optimizers.